In [1]:
collection_name='pib_chunks'
import asyncio

In [ ]:
! pip install qdrant-client

In [2]:
async def create_corpus():
    from rag.ingestion.corpus import iter_chunks_from_sqlite
    from pathlib import Path
    from rag.retrieval.vector_store import store
    vectors:list = []
    i=0
    for chunk in iter_chunks_from_sqlite(Path("rag/data/main_corpus.db")):
        vectors.append(chunk)
        if len(vectors) == 100:
            print(f"Batch:{i}")
            await store.ingest_chunks(vectors)
            vectors.clear()
            i+=1


In [3]:
import os

async def create_vector_collection():
    from qdrant_client import AsyncQdrantClient
    client = AsyncQdrantClient(url=os.getenv("CLUSTER_ENDPOINT"), api_key=os.getenv("CLUSTER_API"), check_compatibility=False)

    if await client.collection_exists(collection_name=collection_name):
        collection_info = await client.get_collection(collection_name=collection_name)
        points_count = collection_info.points_count
        if points_count >= 15000:
            return
        else:
            print("Meow")
            from rag.retrieval.vector_store import entry
            await client.delete_collection("pib_chunks")
            await entry()
    else:
        from rag.retrieval.vector_store import entry
        print("Meow 2")
        await entry()
        await create_corpus()
    return


In [4]:
async def main():
    await create_vector_collection()

await main()

ModuleNotFoundError: No module named 'qdrant_client'